# Task 3 — Noise robustness
Szum Gaussowski w 5 pasmach PSNR. ≥300 obrazów na pasmo.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, cv2
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.degradation import add_noise_to_psnr, PSNR_BANDS, psnr_band_midpoint
from src.metrics import compute_far_frr
from src.utils import list_images, psnr

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()
THRESHOLD = 0.4
TEST_DIR = Path('../data/test')

In [ ]:
all_images = []
for user_dir in TEST_DIR.iterdir():
    if user_dir.name not in db.get_all_users(): continue
    for p in list_images(user_dir):
        img = cv2.imread(str(p))
        if img is not None: all_images.append((user_dir.name, img))
print(f'Obrazów testowych enrolled: {len(all_images)}')

In [ ]:
band_results = {}
for band in PSNR_BANDS:
    target = psnr_band_midpoint(band)
    scores, labels = [], []
    for user_id, img in all_images[:300]:
        noisy = add_noise_to_psnr(img, target)
        emb = get_embedding(app, noisy)
        if emb is None: continue
        refs = db.get_user_embeddings(user_id)
        score = max(cosine_similarity(emb, r) for r in refs)
        scores.append(score); labels.append(1)
    far, frr, acc = compute_far_frr(np.array(scores), np.array(labels), THRESHOLD)
    band_results[f'{band[0]}-{band[1]} dB'] = {'FAR': far, 'FRR': frr, 'Acc': acc, 'n': len(scores)}
    print(f'PSNR {band[0]}-{band[1]} dB | FAR={far:.4f} FRR={frr:.4f} Acc={acc:.4f} (n={len(scores)})')

In [ ]:
import pandas as pd
df = pd.DataFrame(band_results).T
print(df)
df.to_csv('../results/task3/noise_results.csv')

fig, ax = plt.subplots()
ax.plot(df.index, df['FAR'], marker='o', label='FAR')
ax.plot(df.index, df['FRR'], marker='s', label='FRR')
ax.set_xlabel('PSNR Band'); ax.set_ylabel('Rate'); ax.legend()
plt.tight_layout()
plt.savefig('../results/task3/noise_plot.png', dpi=150)
plt.show()